[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlospi2000/Notes_on_Computational_Physics/blob/main/12.Ord_Diff_Equations/EDO_RungeKutta_ESTUDIANTES.ipynb)

# Semana 11b — Introducción a la solución numérica de ecuaciones diferenciales

**Física Computacional — 106018C** · Lenguaje: **Fortran y Python** · Capítulo 7: Aplicaciones en Física

Las leyes de la física casi siempre se expresan como **ecuaciones diferenciales**: dicen cómo **cambia** una cantidad, no cuánto vale. Este notebook presenta cómo se resuelven en el computador, paso a paso en el tiempo, con tres métodos de precisión creciente: Euler, Runge-Kutta de orden 2 y Runge-Kutta de orden 4. Termina con el **Taller 11**.

**Objetivos de este notebook**

1. Plantear un problema de valor inicial y reconocer su solución como una curva que sigue el campo de direcciones.
2. Relacionar cada método de paso con una regla de integración de la Semana 7.
3. Programar los métodos de Euler, RK2 y RK4, y medir su orden de convergencia.
4. Reconocer la inestabilidad del método de Euler cuando el paso es demasiado grande.

In [ ]:
# Preparación: bibliotecas de Python, colores del curso y herramientas de compilación
import time
import numpy as np
import matplotlib.pyplot as plt

AZUL, DORADO, ROJO, VERDE, GRIS, MORADO = "#162A54", "#C9972C", "#B02A20", "#2E7D32", "#7F7F7F", "#6A3D9A"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
np.set_printoptions(precision=4, suppress=True)

# F2PY compila con meson y ninja desde Python 3.12 (igual que en las Semanas 9 y 10)
!pip -q install meson ninja
!gfortran --version | head -n 1
print("NumPy", np.__version__)

## 1. Ecuaciones diferenciales ordinarias y problema de valor inicial

Una **ecuación diferencial ordinaria** (EDO) de primer orden relaciona una función $y(t)$ con su derivada:

$$\frac{dy}{dt} = f(t, y), \qquad y(t_0) = y_0,$$

donde

- $t$ es la variable independiente (en física, casi siempre el tiempo);
- $y(t)$ es la cantidad desconocida (una población, una temperatura, una velocidad);
- $f(t, y)$ es la **tasa de cambio**: la ley física que dice qué tan rápido cambia $y$ en el instante $t$ si su valor es $y$;
- $y(t_0) = y_0$ es la **condición inicial**: el valor de $y$ en el instante inicial.

La ecuación junto con la condición inicial forman un **problema de valor inicial**, que tiene una única solución cuando $f$ es suficientemente suave. Tres ejemplos:

| Fenómeno | Ecuación | Solución analítica |
|---|---|---|
| Decaimiento radiactivo (Semana 1, carbono-14) | $\dfrac{dN}{dt} = -\lambda N$ | $N(t) = N_0\, e^{-\lambda t}$ |
| Enfriamiento de Newton | $\dfrac{dT}{dt} = -k\,(T - T_a)$ | $T(t) = T_a + (T_0 - T_a)\,e^{-kt}$ |
| Caída con arrastre del aire (Taller 11) | $\dfrac{dv}{dt} = g\left(1 - \dfrac{v^2}{v_t^2}\right)$ | $v(t) = v_t \tanh(g t / v_t)$ |

En la tabla, $\lambda$ es la constante de decaimiento, $k$ la constante de enfriamiento, $T_a$ la temperatura del ambiente, $g$ la aceleración de la gravedad y $v_t$ la velocidad terminal. Estas tres tienen solución analítica, y por eso sirven para **verificar** los métodos numéricos. La mayoría de las ecuaciones de interés (un péndulo de gran amplitud, un sistema de muchos cuerpos, un circuito no lineal) no la tienen.

### 1.1 El campo de direcciones

La ecuación $dy/dt = f(t, y)$ asigna una **pendiente** a cada punto del plano $(t, y)$. Si se dibuja un segmento corto con esa pendiente en muchos puntos, se obtiene el **campo de direcciones**. La solución es la curva que, partiendo de $(t_0, y_0)$, es tangente al campo en todos sus puntos. La figura lo muestra para el enfriamiento de una taza de café con $T_a = 20\ ^\circ$C y $k = 0.1\ \text{min}^{-1}$.

In [ ]:
# Figura: campo de direcciones del enfriamiento de Newton y soluciones con distintas T0
k, Ta = 0.1, 20.0                                   # [1/min], [°C]
f_newton = lambda t, T: -k*(T - Ta)

tg, Tg = np.meshgrid(np.linspace(0, 40, 21), np.linspace(0, 100, 21))
pend = f_newton(tg, Tg)
norma = np.sqrt(1 + (pend/2.5)**2)                  # segmentos de igual longitud en la figura

fig, ax = plt.subplots(figsize=(8, 4))
ax.quiver(tg, Tg, 1/norma, (pend/2.5)/norma, color=GRIS, angles="xy", pivot="mid",
          scale=45, width=0.0025, headwidth=0, headlength=0, headaxislength=0)
tt = np.linspace(0, 40, 300)
for T0, c in [(95, ROJO), (60, DORADO), (5, AZUL)]:
    ax.plot(tt, Ta + (T0 - Ta)*np.exp(-k*tt), color=c, lw=2, label=f"T0 = {T0} °C")
    ax.plot(0, T0, "o", color=c)
ax.axhline(Ta, color=VERDE, ls="--", lw=1, label="T_a = 20 °C")
ax.set(xlabel="t [min]", ylabel="T [°C]", title="dT/dt = -k (T - T_a): campo de direcciones y soluciones")
ax.legend(fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

Todas las soluciones tienden a $T_a$, donde la pendiente es cero: $T = T_a$ es un **punto de equilibrio** de la ecuación. El café caliente se enfría y el agua fría se calienta, y lo dice una sola ley, $f(t, T) = -k(T - T_a)$.

## 2. Avanzar paso a paso: la idea de todos los métodos

El computador no obtiene la curva $y(t)$ completa, sino sus valores en instantes discretos separados por un **paso** $h$:

$$t_n = t_0 + n\,h, \qquad y_n \approx y(t_n), \qquad n = 0, 1, 2, \dots$$

Integrando la ecuación entre $t_n$ y $t_{n+1}$ se obtiene una relación **exacta**:

$$y(t_{n+1}) = y(t_n) + \int_{t_n}^{t_{n+1}} f\big(t, y(t)\big)\,dt .$$

Todos los métodos de esta semana tienen la misma forma: el valor siguiente es el valor actual más una **aproximación de la integral** de la pendiente sobre un paso. Lo que los distingue es la regla de integración que usan, y ahí reaparecen las reglas de la Semana 7. La dificultad nueva es que $f$ depende de $y(t)$, que solo se conoce en $t_n$: los valores intermedios también hay que estimarlos.

## 3. El método de Euler

La aproximación más simple de la integral es un **rectángulo** con la pendiente del extremo izquierdo:

$$\int_{t_n}^{t_{n+1}} f\,dt \approx h\, f(t_n, y_n) \qquad\Longrightarrow\qquad \boxed{\;y_{n+1} = y_n + h\, f(t_n, y_n)\;}$$

donde

- $y_n$ es la aproximación de la solución en $t_n$;
- $f(t_n, y_n)$ es la pendiente en el punto actual, dada por la ecuación;
- $h\,f(t_n, y_n)$ es el cambio de $y$ si la pendiente se mantuviera constante durante el paso.

Geométricamente, Euler avanza por la **recta tangente** durante un paso, y en el punto nuevo toma la pendiente que corresponde a ese punto. Es la definición de derivada sin el límite: $f \approx (y_{n+1} - y_n)/h$, la derivada adelantada de la Semana 7.

In [ ]:
# Métodos de un paso: reciben f, el punto actual (t, y) y el paso h; devuelven y en t + h
def paso_euler(f, t, y, h):
    """Euler: avanza con la pendiente del punto actual."""
    return y + h*f(t, y)

def resolver(paso, f, t0, y0, h, n):
    """Aplica n pasos de un método desde (t0, y0). Devuelve los arreglos t y y de n + 1 puntos."""
    t = t0 + h*np.arange(n + 1)
    y = np.zeros(n + 1)
    y[0] = y0
    for i in range(n):
        y[i + 1] = paso(f, t[i], y[i], h)
    return t, y

In [ ]:
# Figura: Euler avanza por la tangente (ecuación de prueba dy/dt = -y, y(0) = 1, solución exp(-t))
f_prueba = lambda t, y: -y
t, y = resolver(paso_euler, f_prueba, 0.0, 1.0, 0.5, 6)
tt = np.linspace(0, 3, 300)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(tt, np.exp(-tt), color=AZUL, lw=2, label="exacta  e^(-t)")
ax.plot(t, y, "o-", color=ROJO, label="Euler, h = 0.5")
for i in range(len(t) - 1):                         # curva exacta que pasa por cada punto de Euler
    ts = np.linspace(t[i], 3, 100)
    ax.plot(ts, y[i]*np.exp(-(ts - t[i])), ":", color=GRIS, lw=1)
ax.set(xlabel="t", ylabel="y", title="Cada paso de Euler sigue la tangente y salta a otra solución")
ax.legend(); plt.tight_layout(); plt.show()
print(f"y(3): Euler {y[-1]:.5f},  exacta {np.exp(-3):.5f}")

Las curvas punteadas son las soluciones exactas que pasan por cada punto de Euler. En cada paso el método se sale de la solución en la que estaba y queda en otra vecina. Esos errores se acumulan.

### 3.1 Error local, error global y orden

- **Error local**: el que se comete en **un** paso, partiendo de un valor exacto. Para Euler es proporcional a $h^2$ (el término de la serie de Taylor que se descarta, $\tfrac{h^2}{2}y''$).
- **Error global**: el error acumulado al llegar a un instante fijo $t$. Se necesitan $n = (t - t_0)/h$ pasos, de modo que el error global es del orden de $n\,h^2 \propto h$.

Un método es de **orden $p$** si su error global es proporcional a $h^p$. Euler es de **orden 1**: al reducir $h$ a la mitad, el error se reduce a la mitad. Es la misma verificación por orden de convergencia de la Semana 7.

In [ ]:
# Error de Euler en t = 1 para dy/dt = -y, al reducir h a la mitad
print("     h        y(1)        error     razón")
error_anterior = None
for h in [0.1, 0.05, 0.025, 0.0125]:
    n = round(1/h)
    t, y = resolver(paso_euler, f_prueba, 0.0, 1.0, h, n)
    error = abs(y[-1] - np.exp(-1))
    razon = "" if error_anterior is None else f"{error_anterior/error:6.2f}"
    print(f"{h:7.4f}   {y[-1]:.7f}   {error:.3e}   {razon}")
    error_anterior = error

### 3.2 Inestabilidad del método de Euler

Para la ecuación $dy/dt = -\lambda y$, con $\lambda > 0$, un paso de Euler es

$$y_{n+1} = y_n - h\lambda\, y_n = (1 - \lambda h)\, y_n \qquad\Longrightarrow\qquad y_n = (1 - \lambda h)^n\, y_0 ,$$

donde $(1 - \lambda h)$ es el **factor de amplificación**: el número por el que se multiplica la solución en cada paso. La solución exacta decae, de modo que la numérica también debe decaer, y eso exige $|1 - \lambda h| < 1$, es decir,

$$h < \frac{2}{\lambda}.$$

Si $1 < \lambda h < 2$, el factor es negativo: la solución decae, pero **cambia de signo** en cada paso, algo que la solución exacta nunca hace. Si $\lambda h > 2$, el factor tiene módulo mayor que 1 y la solución numérica **crece sin límite**, aunque la exacta tienda a cero. No es un problema de redondeo: es el método.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Un resultado que crece o que oscila no es necesariamente física.** Si su solución numérica de un problema que debería decaer crece sin control, o alterna de signo paso a paso, reduzca $h$ antes de interpretar nada. Si el comportamiento desaparece al reducir $h$, era inestabilidad del método.

</div>

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento 2 (no calificado): la inestabilidad de Euler**

Con $dy/dt = -\lambda y$, $\lambda = 10$ y $y(0) = 1$, la solución exacta en $t = 3$ es $e^{-30} \approx 9.36\times10^{-14}$. En la celda siguiente:

1. Resuelva con Euler hasta $t = 3$ con $h = 0.05$ (60 pasos), $h = 0.15$ (20 pasos) y $h = 0.25$ (12 pasos), usando `resolver` y `paso_euler`.
2. Obtenga una figura con las tres soluciones numéricas, en escala lineal, con el eje vertical limitado a $[-2, 2]$.
3. Obtenga $y(3)$ para cada $h$ y compárelo con $(1 - \lambda h)^n$.

**Debe obtener:** $y(3) = 8.67\times10^{-19}$, $9.54\times10^{-7}$ y $129.7$; las dos primeras decaen (la segunda alternando de signo) y la tercera crece, de acuerdo con los factores $0.5$, $-0.5$ y $-1.5$.

</div>

In [ ]:
# Ejercicio de calentamiento 2
lam = 10.0
f_rapida = lambda t, y: -lam*y

# TODO 1: resuelva con Euler para h = 0.05, 0.15 y 0.25 hasta t = 3
# TODO 2: figura con las tres soluciones, eje vertical en [-2, 2]
# TODO 3: imprima y(3) y (1 - lam h)^n para cada h

## 4. Runge-Kutta de orden 2 (punto medio)

La regla del **punto medio** aproxima la integral con la pendiente en la mitad del intervalo, que es mucho más representativa que la del extremo:

$$\int_{t_n}^{t_{n+1}} f\,dt \approx h\, f\!\left(t_n + \tfrac{h}{2},\ y(t_n + \tfrac{h}{2})\right).$$

El problema es que $y(t_n + h/2)$ no se conoce. Se **estima** con medio paso de Euler, y con esa estimación se evalúa la pendiente:

$$
\begin{aligned}
k_1 &= f(t_n,\ y_n), \\
k_2 &= f\!\left(t_n + \tfrac{h}{2},\ y_n + \tfrac{h}{2}\,k_1\right), \\
y_{n+1} &= y_n + h\, k_2 ,
\end{aligned}
$$

donde

- $k_1$ es la pendiente en el punto actual (la de Euler);
- $y_n + \tfrac{h}{2}k_1$ es la estimación de $y$ en la mitad del paso, con medio paso de Euler;
- $k_2$ es la pendiente en esa estimación del punto medio;
- el paso completo se da con $k_2$, no con $k_1$.

Cada paso evalúa $f$ **dos** veces. A cambio, el error local es $O(h^3)$ y el global $O(h^2)$: el método es de **orden 2**. Al reducir $h$ a la mitad, el error se reduce 4 veces.

In [ ]:
def paso_rk2(f, t, y, h):
    """Runge-Kutta de orden 2 (punto medio)."""
    k1 = f(t, y)                           # pendiente en el punto actual
    k2 = f(t + h/2, y + h/2*k1)            # pendiente en la estimación del punto medio
    return y + h*k2

## 5. Runge-Kutta de orden 4

El método más usado en la práctica evalúa la pendiente **cuatro** veces por paso y las promedia con pesos:

$$
\begin{aligned}
k_1 &= f(t_n,\ y_n), \\
k_2 &= f\!\left(t_n + \tfrac{h}{2},\ y_n + \tfrac{h}{2}\,k_1\right), \\
k_3 &= f\!\left(t_n + \tfrac{h}{2},\ y_n + \tfrac{h}{2}\,k_2\right), \\
k_4 &= f\!\left(t_n + h,\ y_n + h\,k_3\right), \\
y_{n+1} &= y_n + \frac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right),
\end{aligned}
$$

donde

- $k_1$ es la pendiente al inicio del paso;
- $k_2$ es la pendiente en el punto medio, estimado con $k_1$ (como en RK2);
- $k_3$ es otra vez la pendiente en el punto medio, pero estimado con $k_2$, que es mejor que $k_1$;
- $k_4$ es la pendiente al final del paso, estimado con $k_3$;
- el promedio pesa el doble las dos estimaciones del punto medio.

**Relación con Simpson.** Si $f$ dependiera solo de $t$, sería $k_2 = k_3 = f(t_n + h/2)$ y el paso quedaría

$$y_{n+1} = y_n + \frac{h}{6}\Big[f(t_n) + 4 f(t_n + \tfrac{h}{2}) + f(t_n + h)\Big],$$

que es exactamente la **regla de Simpson** de la Semana 7 sobre el intervalo $[t_n, t_n + h]$. RK4 es la generalización de Simpson al caso en que la pendiente depende de la solución. Es de **orden 4**: al reducir $h$ a la mitad, el error se reduce 16 veces.

| Método | Regla de integración (Semana 7) | Evaluaciones de $f$ por paso | Orden |
|---|---|---|---|
| Euler | rectángulo izquierdo | 1 | 1 |
| RK2 | punto medio | 2 | 2 |
| RK4 | Simpson | 4 | 4 |

In [ ]:
def paso_rk4(f, t, y, h):
    """Runge-Kutta de orden 4."""
    k1 = f(t, y)                           # pendiente al inicio
    k2 = f(t + h/2, y + h/2*k1)            # punto medio, estimado con k1
    k3 = f(t + h/2, y + h/2*k2)            # punto medio, estimado con k2
    k4 = f(t + h, y + h*k3)                # final del paso, estimado con k3
    return y + h/6*(k1 + 2*k2 + 2*k3 + k4)

In [ ]:
# Figura: un solo paso grande (h = 1) de cada método para dy/dt = -y, y(0) = 1
tt = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(tt, np.exp(-tt), color=AZUL, lw=2, label=f"exacta: y(1) = {np.exp(-1):.4f}")
for paso, nombre, c in [(paso_euler, "Euler", ROJO), (paso_rk2, "RK2", DORADO), (paso_rk4, "RK4", VERDE)]:
    y1 = paso(f_prueba, 0.0, 1.0, 1.0)
    ax.plot([0, 1], [1, y1], "o--", color=c, label=f"{nombre}: y(1) = {y1:.4f}")
ax.set(xlabel="t", ylabel="y", title="Un paso de h = 1 con cada método"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 6. Orden de convergencia de los tres métodos

La verificación definitiva del orden es la de la Semana 7: resolver la ecuación de prueba hasta $t = 1$ con pasos cada vez más pequeños y observar cómo cae el error. En escala log-log, el error de un método de orden $p$ es una recta de pendiente $p$.

In [ ]:
metodos = [(paso_euler, "Euler", ROJO, 1), (paso_rk2, "RK2", DORADO, 2), (paso_rk4, "RK4", VERDE, 4)]
pasos = np.array([0.1, 0.05, 0.025, 0.0125, 0.00625])
errores = {}
for paso, nombre, _, _ in metodos:
    errores[nombre] = []
    for h in pasos:
        t, y = resolver(paso, f_prueba, 0.0, 1.0, h, round(1/h))
        errores[nombre].append(abs(y[-1] - np.exp(-1)))
    e = np.array(errores[nombre])
    print(f"{nombre:5s}: errores {' '.join(f'{x:.2e}' for x in e)}   razones {' '.join(f'{r:5.2f}' for r in e[:-1]/e[1:])}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for paso, nombre, c, p in metodos:
    e = np.array(errores[nombre])
    ax[0].loglog(pasos, e, "o-", color=c, label=f"{nombre} (orden {p})")
    ax[0].loglog(pasos, e[0]*(pasos/pasos[0])**p, ":", color=c, alpha=0.6)
    evaluaciones = p*np.round(1/pasos)            # p evaluaciones de f por paso, hasta t = 1
    ax[1].loglog(evaluaciones, e, "o-", color=c, label=nombre)
ax[0].set(xlabel="paso h", ylabel="error en t = 1", title="Error frente al paso (punteado: pendiente p)")
ax[1].set(xlabel="evaluaciones de f", ylabel="error en t = 1", title="Error frente al trabajo")
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

Las razones se acercan a 2, 4 y 16, y en la gráfica log-log los errores siguen rectas de pendiente 1, 2 y 4. La gráfica de la derecha compara los métodos con el **mismo trabajo**: aunque RK4 evalúa $f$ cuatro veces por paso, para cualquier precisión razonable necesita muchas menos evaluaciones en total que Euler. Por eso RK4 es el método de referencia para problemas que no son muy exigentes.

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

**El redondeo pone un piso.** Si se sigue reduciendo $h$, el error de RK4 deja de bajar alrededor de $10^{-14}$–$10^{-15}$ y luego sube lentamente, porque se suman muchos errores de redondeo de muchos pasos. Es el mismo compromiso entre truncamiento y redondeo que se vio con las derivadas numéricas en la Semana 7.

</div>

## 7. Los métodos en Fortran

En Fortran, la ecuación se programa como una **función** dentro de un módulo, y el método llama a esa función en cada etapa. El programa siguiente resuelve la ecuación de prueba con RK4 y es la estructura del ejercicio de calentamiento 3 y del Taller 11.

In [ ]:
%%writefile prueba_rk4.f90
! ============================================================
! Programa:  prueba_rk4
! Propósito: resolver dy/dt = -y, y(0) = 1, hasta t = 1 con
!            RK4 y comparar con la solución exacta exp(-1).
! Salida:    en pantalla
! ============================================================
module ecuacion
  implicit none
  integer, parameter :: dp = kind(1.0d0)
contains
  ! f: tasa de cambio dy/dt = f(t, y)
  function f(t, y) result(pendiente)
    real(dp), intent(in) :: t, y
    real(dp) :: pendiente
    pendiente = -y
  end function f

  ! paso_rk4: avanza un paso h desde (t, y) con Runge-Kutta de orden 4
  function paso_rk4(t, y, h) result(y_nuevo)
    real(dp), intent(in) :: t, y, h
    real(dp) :: y_nuevo, k1, k2, k3, k4
    k1 = f(t, y)
    k2 = f(t + h/2, y + h/2*k1)
    k3 = f(t + h/2, y + h/2*k2)
    k4 = f(t + h, y + h*k3)
    y_nuevo = y + h/6*(k1 + 2*k2 + 2*k3 + k4)
  end function paso_rk4
end module ecuacion

program prueba_rk4
  use ecuacion
  implicit none
  real(dp) :: t, y, h
  integer :: n, i

  h = 0.1_dp                                ! paso
  n = 10                                    ! número de pasos hasta t = 1
  t = 0.0_dp
  y = 1.0_dp                                ! condición inicial
  do i = 1, n
     y = paso_rk4(t, y, h)                  ! primero se avanza y con el t actual
     t = t + h                              ! y después el tiempo
  end do
  print '(a,f12.9)', 'y(1) con RK4 = ', y
  print '(a,es10.3)', 'error        = ', abs(y - exp(-1.0_dp))
end program prueba_rk4

In [ ]:
!gfortran -fcheck=bounds -o prueba_rk4 prueba_rk4.f90 && ./prueba_rk4

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**El orden de las actualizaciones importa.** Dentro del ciclo, $y$ se actualiza con el $t$ del **inicio** del paso, y solo después se avanza $t$. Si se invierte el orden, todas las pendientes se evalúan en el instante equivocado; con $f(t, y) = -y$ no se nota, porque $f$ no depende de $t$, pero con una ecuación que dependa del tiempo el método pierde su orden.

</div>

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento 3 (no calificado): una taza de café en Fortran**

Una taza de café a $T_0 = 90\ ^\circ$C se enfría en un ambiente a $T_a = 20\ ^\circ$C según la ley de Newton, $dT/dt = -k(T - T_a)$, con $k = 0.1\ \text{min}^{-1}$. La solución exacta en $t = 10$ min es $T_a + (T_0 - T_a)e^{-1} = 45.751561\ ^\circ$C.

En `cafe.f90`, con la estructura del programa `prueba_rk4`:

1. Programe la función $f(t, T)$ del enfriamiento de Newton y las funciones de un paso de Euler y de RK4.
2. Con $h = 1$ min, obtenga $T(10)$ con cada método y su error frente a la solución exacta.

**Debe obtener:** con Euler, $T(10) = 44.407491\ ^\circ$C, con un error de $1.34\ ^\circ$C; con RK4, $T(10) = 45.751584\ ^\circ$C, con un error de $2.3\times10^{-5}\ ^\circ$C: con el mismo paso, RK4 es unas 60 000 veces más preciso.

</div>

In [ ]:
%%writefile cafe.f90
! ============================================================
! Programa:  cafe   (Ejercicio de calentamiento 3)
! Propósito: enfriamiento de Newton con Euler y con RK4.
! Salida:    en pantalla
! ============================================================
module enfriamiento
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  real(dp), parameter :: k = 0.1_dp           ! constante de enfriamiento [1/min]
  real(dp), parameter :: ta = 20.0_dp         ! temperatura ambiente [°C]
contains
  ! TODO 1: función f(t, temp) del enfriamiento de Newton
  ! TODO 1: funciones paso_euler(t, temp, h) y paso_rk4(t, temp, h)
end module enfriamiento

program cafe
  use enfriamiento
  implicit none
  real(dp), parameter :: h = 1.0_dp           ! paso [min]
  integer,  parameter :: n = 10               ! pasos hasta t = 10 min
  real(dp) :: t, te, tr, exacta
  integer :: i

  exacta = ta + (90.0_dp - ta)*exp(-k*10.0_dp)
  ! TODO 2: obtenga T(10) con Euler y con RK4 (h = 1 min) e imprima cada valor y su error
end program cafe

In [ ]:
!gfortran -fcheck=bounds -o cafe cafe.f90 && ./cafe

## 8. Taller 11: caída con arrastre del aire

Un paracaidista salta desde un avión y cae, antes de abrir el paracaídas, con una fuerza de arrastre del aire proporcional al cuadrado de su velocidad. Con la dirección hacia abajo como positiva, la segunda ley de Newton da

$$\frac{dv}{dt} = g\left(1 - \frac{v^2}{v_t^2}\right), \qquad v(0) = 0,$$

donde

- $v(t)$ es la velocidad de caída [m/s];
- $g = 9.81\ \text{m/s}^2$ es la aceleración de la gravedad;
- $v_t = 50\ \text{m/s}$ es la **velocidad terminal**: la velocidad a la que el arrastre iguala al peso y la aceleración se anula;
- el término $g\,v^2/v_t^2$ es la desaceleración por el arrastre, que crece con el cuadrado de la velocidad.

La ecuación no es lineal en $v$, pero tiene solución analítica, que sirve para verificar:

$$v(t) = v_t \tanh\!\left(\frac{g\,t}{v_t}\right).$$

### A. Enunciado y requisitos

**Requisitos**

1. Las celdas del taller trabajan dentro de la carpeta `taller_edo`, que crea la celda de la Parte B. En el repositorio, la carpeta `taller_edo` contiene `modulo_edo.f90` y este notebook.
2. El módulo se llama `edo`, está en `modulo_edo.f90` y se compila como la librería `edof`:
   `python -m numpy.f2py -c modulo_edo.f90 -m edof`
3. El módulo define $g$ y $v_t$ como parámetros (`parameter`) y contiene las tres rutinas siguientes, que la celda de verificación llama directamente:

| Rutina | Entradas | Salida | Uso desde Python |
|---|---|---|---|
| función `f` | `t`, `v` | $g\,(1 - v^2/v_t^2)$ | `edof.edo.f(t, v)` |
| función `paso` | `metodo`, `t`, `v`, `h` | $v$ en $t + h$; `metodo` = 1 Euler, 2 RK2, 3 RK4 | `edof.edo.paso(metodo, t, v, h)` |
| subrutina `resolver` | `metodo`, `h`, `n`, `v0` | arreglos `t(n+1)` y `v(n+1)` desde $t = 0$ | `t, v = edof.edo.resolver(metodo, h, n, v0)` |

4. `paso` llama a `f` en cada etapa del método, y `resolver` llama a `paso` en cada paso. Ninguna de las tres usa la solución analítica.
5. Se usan `real(8)` e `integer`. El módulo no usa `stop` ni imprime en pantalla.

**Qué se entrega**

- En la carpeta del repositorio: `modulo_edo.f90` y este notebook, con la celda de verificación en 3 de 3 PASS y las respuestas de la sección D en celdas Markdown al final.
- En Moodle: el reporte de una página y el enlace a la carpeta.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `scipy.integrate` (`solve_ivp`, `odeint`) ni las funciones de Python de este notebook (`paso_euler`, `paso_rk2`, `paso_rk4`, `resolver`) dentro de las actividades. Las soluciones numéricas se obtienen con su librería de Fortran; la solución analítica solo se usa para comparar.

</div>

### B. Parte Fortran: la librería

Ejecute primero la celda siguiente, que crea la carpeta `taller_edo` y la deja como carpeta de trabajo. Después escriba `modulo_edo.f90` y compílelo. Recuerde de la Semana 9: después de cada recompilación, **reinicie el entorno** y ejecute de nuevo la celda de preparación y esta celda.

In [ ]:
# Carpeta de trabajo del taller: aquí se escriben, compilan, importan y verifican los archivos
import os, sys
CARPETA = "taller_edo"
if os.path.basename(os.getcwd()) != CARPETA:     # evita crear taller_edo/taller_edo
    os.makedirs(CARPETA, exist_ok=True)
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())               # import edof busca la librería aquí
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
# Compila la librería "edof" (ejecute esta celda cuando su archivo esté listo)
!python -m numpy.f2py -c modulo_edo.f90 -m edof 2>&1 | tail -n 3

### C. Parte Python: uso de la librería

Ejecute las actividades en las celdas siguientes. Cada una indica qué debe obtener. En todas, $v(0) = 0$.

**Actividad 1. Prueba de la librería.** Importe `edof` e imprima la documentación de `resolver`. Obtenga:

- $f(0, 0)$ y $f(0, v_t)$, que deben ser $9.81$ y $0$;
- un paso de cada método desde $t = 0$, $v = 0$, con $h = 1$ s, que debe dar $9.810000$ (Euler), $9.715592$ (RK2) y $9.685909$ (RK4).

**Actividad 2. Los tres métodos con un paso grande.** Con $h = 1$ s y $0 \le t \le 30$ s, obtenga:

- una figura con las tres soluciones numéricas y la analítica, y la línea horizontal $v = v_t$;
- una tabla con $v(10)$ de cada método y su error frente a la solución analítica, $v(10) = 48.062108$ m/s. Debe obtener errores de $0.92$ (Euler), $0.11$ (RK2) y $9.5\times10^{-4}$ m/s (RK4).

**Actividad 3. Orden de convergencia.** Para $h = 0.4, 0.2, 0.1$ y $0.05$ s, resuelva hasta $t = 10$ s con los tres métodos. Obtenga:

- una tabla con el error en $t = 10$ s para cada método y cada $h$, y la razón entre errores consecutivos;
- una figura log-log del error frente a $h$ para los tres métodos.

Debe obtener razones cercanas a 2 (Euler), 4 (RK2) y 16 (RK4), y rectas de pendiente 1, 2 y 4.

**Actividad 4. Tiempo para alcanzar el 99 % de la velocidad terminal.** Con RK4 y $h = 0.1$ s hasta $t = 30$ s, obtenga:

- el error máximo frente a la solución analítica en todo el intervalo, que debe ser menor que $10^{-6}$ m/s;
- el instante $t_{99}$ en que $v = 0.99\,v_t$, por interpolación lineal entre los dos pasos que lo encierran, y su diferencia con el valor analítico $t_{99} = (v_t/g)\,\mathrm{artanh}(0.99) = 13.4896$ s, que debe ser menor que $0.001$ s.

In [ ]:
# Actividad 1

In [ ]:
# Actividad 2

In [ ]:
# Actividad 3

In [ ]:
# Actividad 4

### D. Preguntas de interpretación

Responda en celdas Markdown al final de este notebook, con los números de sus resultados:

1. **Equilibrio.** ¿Por qué, en la solución analítica, la velocidad nunca supera $v_t$? Explíquelo con el signo de $f(t, v)$ a cada lado de $v = v_t$, como en el campo de direcciones de la sección 1.1.
2. **Costo.** Con sus errores de la actividad 3, estime cuántas evaluaciones de $f$ necesita cada método para obtener $v(10)$ con un error menor que $10^{-4}$ m/s. ¿Cuál conviene usar, y por qué el número de evaluaciones por paso no es lo que decide?
3. **Estabilidad.** Resuelva con Euler y $h = 6$ s hasta $t = 60$ s. Describa lo que obtiene. Cerca de $v_t$, la ecuación se comporta como $dy/dt = -\lambda y$ para la diferencia $y = v - v_t$, con $\lambda = 2g/v_t$. Use el criterio $h < 2/\lambda$ de la sección 3.2 para explicar su resultado.

### E. Verificación

La celda siguiente **no debe modificarse**. Importa su librería y la prueba con casos de resultado conocido. Si recompiló la librería, reinicie el entorno antes de ejecutarla. El taller exige los tres PASS.

In [ ]:
# ==================== VERIFICACIÓN DEL TALLER 11 (no modificar) ====================
import os, sys, glob
import numpy as np

resultados = []
def caso(nombre, funcion):
    try:
        detalle = funcion()
        ok = True
    except Exception as e:
        ok, detalle = False, str(e)
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ""))

# La verificación siempre trabaja dentro de la carpeta del taller
CARPETA = "taller_edo"
if os.path.basename(os.getcwd()) != CARPETA:
    if not os.path.isdir(CARPETA):
        raise SystemExit(f"No existe la carpeta {CARPETA}: ejecute la celda de la Parte B.")
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Verificando en:", os.getcwd(), "\n")

def libreria():
    if not os.path.isfile("modulo_edo.f90"):
        raise ValueError(f"no existe modulo_edo.f90 en {CARPETA}")
    so = glob.glob("edof.*.so") + glob.glob("edof.*.pyd")
    if not so:
        raise ValueError(f"no se encontró la librería edof en {CARPETA}; compile con F2PY")
    so = max(so, key=os.path.getmtime)                # la más reciente, si hay varias
    if os.path.getmtime(so) < os.path.getmtime("modulo_edo.f90"):
        raise ValueError("la librería es anterior a modulo_edo.f90; recompile y reinicie")
    import edof
    if os.path.abspath(edof.__file__) != os.path.abspath(so):
        raise ValueError(f"la librería cargada es {edof.__file__}, no la de {CARPETA}; reinicie el entorno")
    if not hasattr(edof, "edo"):
        raise ValueError("la librería no contiene el módulo edo")
    return edof.edo

G, VT = 9.81, 50.0
def v_exacta(t):
    return VT*np.tanh(G*np.asarray(t)/VT)

def c1():
    lib = libreria()
    if abs(lib.f(0.0, 0.0) - G) > 1e-12 or abs(lib.f(3.0, VT)) > 1e-12 or abs(lib.f(0.0, 25.0) - 0.75*G) > 1e-12:
        raise ValueError("f(t, v) no es g (1 - v^2/vt^2) con g = 9.81 y vt = 50")
    esperado = {1: 9.81, 2: 9.7155923859, 3: 9.685908539341343}
    for m, v1 in esperado.items():
        if abs(lib.paso(m, 0.0, 0.0, 1.0) - v1) > 1e-9:
            raise ValueError(f"un paso del método {m} desde v = 0 con h = 1 no da {v1:.6f}")
    if abs(lib.paso(3, 2.0, 10.0, 0.5) - 14.6037109191) > 1e-6:
        raise ValueError("un paso de RK4 desde (t, v) = (2, 10) con h = 0.5 es incorrecto")
    return "f y un paso de Euler, RK2 y RK4 correctos"

def c2():
    lib = libreria()
    t, v = lib.resolver(3, 0.1, 300, 0.0)
    if len(t) != 301 or abs(t[0]) > 1e-14 or abs(t[-1] - 30.0) > 1e-9 or abs(v[0]) > 1e-14:
        raise ValueError("resolver no devuelve 301 puntos de t = 0 a t = 30 con v(0) = 0")
    error = np.max(np.abs(v - v_exacta(t)))
    if error > 1e-6:
        raise ValueError(f"error máximo de RK4 con h = 0.1: {error:.2e} (debe ser < 1e-6)")
    t5, v5 = lib.resolver(3, 0.5, 4, 20.0)
    if abs(v5[-1] - v_exacta(2.0 + VT/G*np.arctanh(20.0/VT))) > 1e-4:
        raise ValueError("resolver no respeta la condición inicial v0")
    return f"RK4 con h = 0.1 en [0, 30] s: error máximo {error:.1e} m/s"

def c3():
    lib = libreria()
    rangos = {1: (1.8, 2.2), 2: (3.6, 4.6), 3: (14.0, 19.0)}
    razones = []
    for m, (a, b) in rangos.items():
        e1 = abs(lib.resolver(m, 0.2, 50, 0.0)[1][-1] - v_exacta(10.0))
        e2 = abs(lib.resolver(m, 0.1, 100, 0.0)[1][-1] - v_exacta(10.0))
        r = e1/e2
        razones.append(r)
        if not (a < r < b):
            raise ValueError(f"razón de errores del método {m} al pasar de h = 0.2 a 0.1: {r:.2f} (esperada entre {a} y {b})")
    return "razones " + ", ".join(f"{r:.2f}" for r in razones)

caso("1. Función f y un paso de cada método", c1)
caso("2. Solución con RK4 frente a la analítica", c2)
caso("3. Orden de convergencia de Euler, RK2 y RK4", c3)
print(f"\nResultado: {sum(resultados)} de {len(resultados)} PASS")